In [248]:
import os
import re
import json
from datetime import datetime
import random
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from rag_pipeline.query_engine import load_data, vectorstore, ask_question, is_supported_file

In [4]:
vector_db = "/chroma_db"
session_id = 'test_session'

In [5]:
from ragas import RunConfig
from ragas.testset import TestsetGenerator   
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

In [6]:
generator_llm = LangchainLLMWrapper(
                ChatOllama(
                    model="qwen2.5",
                    temperature=0,
                    # model_kwargs={"response_format":{"type":"json_object"}}
                    format="json"
                    )
                )

generator_embeddings = LangchainEmbeddingsWrapper(
                HuggingFaceEmbeddings(
                    model_name="sentence-transformers/all-MiniLM-L6-v2"
                    )
                )           

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1266.97it/s]


In [7]:
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
run_confing = RunConfig(max_workers=2, timeout=180)

In [8]:
from eval import prepare_testset_documents

In [11]:
docs = prepare_testset_documents("eval_data")

Successfully parsed SouthernStarEnergyInc_20051202_SB-2A_EX-9_801890_EX-9_Affiliate Agreement.pdf
Successfully parsed GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10.1_Content License Agreement.pdf
Successfully parsed TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-10.1_Affiliate Agreement.pdf
Successfully parsed EbixInc_20010515_10-Q_EX-10.3_4049767_EX-10.3_Co-Branding Agreement.pdf
Successfully parsed AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf
Successfully parsed SalesforcecomInc_20171122_10-Q_EX-10.1_10961535_EX-10.1_Reseller Agreement.pdf
Successfully parsed IpassInc_20181203_8-K_EX-99.1_11445874_EX-99.1_Reseller Agreement.pdf
Successfully parsed TodosMedicalLtd_20190328_20-F_EX-4.10_11587157_EX-4.10_Marketing Agreement_ Reseller Agreement.pdf
Successfully parsed ArcGroupInc_20171211_8-K_EX-10.1_10976103_EX-10.1_Sponsorship Agreement.pdf
Successfully parsed hts_2024_revision_10_csv.csv
Successfully parsed Electric_Vehicle_Populat

In [12]:
gen_docs = [file for file in docs if not file.metadata['filename'].endswith('.csv')]

random_docs = random.sample(gen_docs, 200)

In [13]:
lengths = [len(doc.page_content) for doc in random_docs]

print("Docs:", len(lengths))
print("Min:", min(lengths))
print("Max:", max(lengths))
print("Avg:", sum(lengths) / len(lengths))

Docs: 200
Min: 13
Max: 1000
Avg: 700.075


In [15]:
testset = generator.generate_with_langchain_docs(random_docs, testset_size=100, run_config=run_confing)

Applying CustomNodeFilter:   0%|          | 0/200 [00:00<?, ?it/s]               Node 935207cd-f940-4de7-84d3-f99e0cfcb18c does not have a summary. Skipping filtering.
Node c841ae53-0ede-4b1c-8977-83ad7022f11b does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  16%|█▌        | 31/200 [03:09<14:58,  5.32s/it]Node 59ee5126-236e-4816-8a87-b7355ef7f4d5 does not have a summary. Skipping filtering.
Node 31629291-794b-497a-bcdd-0f5da13d1e04 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  24%|██▍       | 49/200 [05:01<19:54,  7.91s/it]Node e80b4a70-3b1b-4ef1-a03d-a4076134455b does not have a summary. Skipping filtering.
Node fa58dcb0-5dbd-410c-9e62-50ac77c4d8dd does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  30%|██▉       | 59/200 [06:05<19:32,  8.32s/it]Node ae48f0f8-f830-48f6-94d9-aade3ab30666 does not have a summary. Skipping filtering.
Node a6fa471c-0808-4da3-bddd-35d211b672d0 does not have a summary. Skipping filtering

In [23]:
df = testset.to_pandas()

In [140]:
testset.to_pandas().to_json("test_data/testset_100.json", orient="records", indent=2)

In [35]:
vectorstore_db = vectorstore(persist_directory="/Volumes/LaCie/Projects_portfolio/IntelliQA/chroma_db", 
                                     documents=docs)

Adding documents to existing vector space


In [430]:
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    ResponseRelevancy,
)
from langchain_google_genai import ChatGoogleGenerativeAI

session_id = "test_session"
golden_set = "eval_data/testset_100.json"

In [431]:
evaluator_llm = LangchainLLMWrapper(
                    ChatGoogleGenerativeAI(
                        model="gemini-flash-lite-latest",
                        temperature=0,
                        # model_kwargs={"response_format": {"type": "json_object"}},
                    )
                )
evaluator_embeddings = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 490.27it/s]


In [432]:
metrics = [
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
]

In [156]:
def get_rag_response(question: str, vectorstore_db, session_id: str)->dict:
    response = ask_question(question, vectorstore_db, session_id, eval=True)
    answer = response['result']
    contexts = [doc.page_content for doc in response['source_documents']]
    return {"answer": answer, "contexts": contexts}

In [532]:
def save_checkpoints(data, file_path):
    
    with open(file_path, 'a') as f:
        f.write(json.dumps(data) + '\n')

In [ ]:
def build_data(df, vectorstore_db, session_id):

    for i, (_, row) in enumerate(df.iterrows(), start=1):
        try:
            response = get_rag_response(row.user_input, vectorstore_db, session_id)

            result = {
                "user_input": row.user_input,
                "retrieved_contexts": response["contexts"],
                "response": response["answer"],
                "reference": row.reference
            }

            save_checkpoints(result, "test_data/rag_results.json")
            print(f"Row {i} processed and saved.")

        except Exception as e:
            print("Current time:", datetime.now().strftime("%H:%M:%S"))
            print("Try after ", re.search(r'(\d+m+\d.)', str(e)).group(0))
            print(f"Error processing row {i}: {e}")
            break
    
    return data

In [312]:
run_config = RunConfig(max_workers=2, timeout=180)

In [521]:
test = build_data(df[25:], vectorstore_db, session_id)

Row 1 processed and saved.
Row 2 processed and saved.
Current time: 18:43:00
Try after  6m7.
Error processing row 3: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jsngve0ge9q95j543y8zf1ma` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99886, Requested 539. Please try again in 6m7.2s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


In [523]:
print(test[-1]['user_input'])
print(df.iloc[27].user_input)

Who is Cao J in the context provided?
What issues might arise from having technical infrastructure outside the United States?


In [443]:
results_v2 = evaluate(dataset=EvaluationDataset.from_list([test[0]]), metrics=metrics, 
                   run_config=run_config, raise_exceptions=True)

Evaluating: 100%|██████████| 4/4 [00:32<00:00,  8.25s/it]


In [445]:
results_v2

{'llm_context_precision_with_reference': 1.0000, 'context_recall': 1.0000, 'faithfulness': 0.4737, 'answer_relevancy': 0.9456}

In [524]:
def load_test_data():
    with open("test_data/rag_results.json", "r") as f:
        return json.load(f)

In [530]:
test = load_test_data()

In [531]:
len(test)

27

In [527]:
df_copy = pd.DataFrame(test)

In [528]:
df_clean = df_copy.drop_duplicates(subset=["user_input"], keep="first").reset_index(drop=True)

In [529]:
df_clean.to_json("test_data/rag_results.json", orient="records", indent=2)